In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from xgboost import XGBClassifier
import torch
import ezkl
import os
from torch import nn
from hummingbird.ml import convert
import hummingbird.ml

np.random.seed(29)

In [2]:
odroid1 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid1_firefox_onscreen.csv")
odroid2 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid2_firefox_onscreen.csv")
odroid3 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid3_firefox_onscreen.csv")
odroid4 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid4_firefox_onscreen.csv")

In [3]:
rpi4a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-4a_chrome_onscreen.csv")
rpi8a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8a_chrome_onscreen.csv")
rpi8b = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8b_chrome_onscreen.csv")
rpi8c = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8c_chrome_onscreen.csv")
rpi8d = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8d_chrome_onscreen.csv")

In [4]:
df = pd.concat([odroid1[:2000], odroid2[:2000], odroid3[:2000], odroid4[:2000], rpi4a[:2000], rpi8a[:2000], rpi8b[:2000], rpi8c[:2000], rpi8d[:2000]])
df = df.sample(frac=1).reset_index(drop=True)

In [5]:
df.head()

,label,Feature 0,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6
0,odroid2,186.0,10.0,150.0,350.0,167.0,167.0,166.0
1,rpi-4a,16.2,17.4,1446.0,17.8,435.5,17.1,19.3
2,odroid1,165.0,167.0,167.0,168.0,169.0,166.0,168.0
3,odroid4,169.0,175.0,176.0,174.0,174.0,173.0,173.0
4,rpi-8c,24.9,629.1,16.7,2092.1,16.1,17.3,17.6


# Data Prep and model training

In [6]:
encoder = LabelEncoder()
encoded_labels = pd.DataFrame(encoder.fit_transform(df['label']))

In [7]:
label_map = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

In [8]:
label_map 


{'odroid1': np.int64(0),
 'odroid2': np.int64(1),
 'odroid3': np.int64(2),
 'odroid4': np.int64(3),
 'rpi-4a': np.int64(4),
 'rpi-8a': np.int64(5),
 'rpi-8b': np.int64(6),
 'rpi-8c': np.int64(7),
 'rpi-8d': np.int64(8)}

In [9]:
df.drop(columns='label', axis=1, inplace=True)

In [10]:
df.head()

,Feature 0,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6
0,186.0,10.0,150.0,350.0,167.0,167.0,166.0
1,16.2,17.4,1446.0,17.8,435.5,17.1,19.3
2,165.0,167.0,167.0,168.0,169.0,166.0,168.0
3,169.0,175.0,176.0,174.0,174.0,173.0,173.0
4,24.9,629.1,16.7,2092.1,16.1,17.3,17.6


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_features = pd.DataFrame(scaler.fit_transform(df))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    scaled_features, encoded_labels, test_size=0.66, shuffle=False
)

In [13]:
X_train

,0,1,2,3,4,5,6
0,0.039312,-0.661995,-0.365168,0.422885,-0.017921,0.086149,0.038358
1,-0.604089,-0.636635,3.175082,-0.745087,1.250471,-0.745914,-0.753278
2,-0.040261,-0.123961,-0.318730,-0.217004,-0.008473,0.080598,0.049150
3,-0.025104,-0.096546,-0.294144,-0.195908,0.015147,0.119454,0.076132
4,-0.571123,1.459639,-0.729300,6.547882,-0.730772,-0.744803,-0.762451
...,...,...,...,...,...,...,...
6115,-0.044050,-0.123961,-0.318730,-0.220520,-0.022645,0.080598,0.043754
6116,-0.044050,-0.089692,-0.348778,-0.202940,-0.003749,0.108352,0.054546
6117,-0.608636,-0.632866,1.916873,-0.749306,3.161326,-0.710944,-0.765689
6118,-0.032682,-0.041714,-0.742139,-0.283805,0.766261,0.086149,0.130094


In [14]:
clf = XGBClassifier()

In [15]:
clf.fit(X_train.values, y_train.values)

# Predict the value of the digit on the test subset
predicted = clf.predict(X_test.values)

In [16]:
print(metrics.classification_report(y_test, predicted))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95      1329
           1       0.98      0.99      0.98      1311
           2       0.95      0.93      0.94      1328
           3       0.99      0.98      0.99      1305
           4       0.85      0.86      0.86      1333
           5       0.87      0.91      0.89      1301
           6       0.89      0.87      0.88      1317
           7       0.70      0.73      0.72      1309
           8       0.74      0.69      0.72      1347

    accuracy                           0.88     11880
   macro avg       0.88      0.88      0.88     11880
weighted avg       0.88      0.88      0.88     11880



## Convert Sklearn -> pytorch

In [17]:
# convert to torch
torch_gbt = convert(clf, 'torch')

# Convert the DataFrame to a numpy array and keep column names
X_test_array = X_test.values

print(torch_gbt)
# assert predictions from torch are = to sklearn
diffs = []

for i in range(len(X_test_array)):
    torch_pred = torch_gbt.predict(torch.tensor(X_test_array[i].reshape(1, -1)))
    sk_pred = clf.predict(X_test_array[i].reshape(1, -1))
    diffs.append(torch_pred != sk_pred[0])

print("num diff: ", sum(diffs))

num diff:  [0]


# EZKL Starts Here:

In [18]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')

witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')

In [19]:
# export to onnx format

# Input to the model
shape = X_train.shape[1:]
x = torch.rand(1, *shape, requires_grad=False)
torch_out = torch_gbt.predict(x)
# Export the model
torch.onnx.export(torch_gbt.model,               # model being run
                  # model input (or a tuple for multiple inputs)
                  x,
                  # where to save the model (can be a file or file-like object)
                  "network.onnx",
                  export_params=True,        # store the trained parameter weights inside the model file
                  opset_version=18,          # the ONNX version to export the model to
                  input_names=['input'],   # the model's input names
                  output_names=['output'],  # the model's output names
                  dynamic_axes={'input': {0: 'batch_size'},    # variable length axes
                                'output': {0: 'batch_size'}})

d = ((x).detach().numpy()).reshape([-1]).tolist()

data = dict(input_shapes=[shape],
            input_data=[d],
            output_data=[(o).reshape([-1]).tolist() for o in torch_out])

# Serialize data into file:
json.dump(data, open("input.json", 'w'))



In [20]:
run_args = ezkl.PyRunArgs()
run_args.variables = [("batch_size", 1)]

# TODO: Dictionary outputs
res = ezkl.gen_settings(model_path, settings_path, py_run_args=run_args)
assert res == True

In [21]:
# generate a bunch of dummy calibration data
cal_data = {
    "input_data": [(torch.rand(20, *shape)).flatten().tolist()],
}

cal_path = os.path.join('calibration.json')
# save as json file
with open(cal_path, "w") as f:
    json.dump(cal_data, f)

res = await ezkl.calibrate_settings(cal_path, model_path, settings_path, "resources")





 <------------- Numerical Fidelity Report (input_scale: 13, param_scale: 13, scale_input_multiplier: 10) ------------->

+-----------------+--------------+---------------+----------------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| mean_error      | median_error | max_error     | min_error      | mean_abs_error | median_abs_error | max_abs_error | min_abs_error | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+-----------------+--------------+---------------+----------------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| 0.0000010001988 | 0            | 0.00024080276 | -0.00023800135 | 0.000033257747 | 0                | 0.00024080276 | 0             | 0.0000000028776992 | 0.0020087843       | 0.023405218            |
+-----------------+--------------+---------------+---------------

In [25]:
# srs path
# res = await ezkl.get_srs( settings_path)

In [26]:
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res == True

In [27]:
# 30s 6GB ram
# run_args.input_scale = 7, run_args.logrows = 10

# 4GB with resources/col-overflow in calibrate
res = ezkl.setup(
        compiled_model_path,
        vk_path,
        pk_path,  
    )

assert res == True
assert os.path.isfile(vk_path) # verification key
assert os.path.isfile(pk_path) # proving key
assert os.path.isfile(settings_path) # circuit settings

PanicException: Once panicked

In [28]:
# now generate the witness file 


res = await ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.isfile(witness_path)

In [29]:
proof_path = os.path.join('test.pf')


# 18s
# NEEDED ~7GB FOR XGBOOST run_args.input_scale = 7

res = ezkl.prove(
        witness_path,
        compiled_model_path,
        pk_path,
        proof_path,
        
        "single",
    )

assert os.path.isfile(proof_path)

RuntimeError: Failed to run prove: [pfsys] failed to load pk from file: The system cannot find the file specified. (os error 2)

In [28]:
# VERIFY IT
# NEEDED ALMOST NO RESOURCES FOR XGBOOST

# The inputs to (non-EVM) verify are:

# the proof file
# the verification key
# the circuit settings, and
# the structured reference string


res = ezkl.verify(
        proof_path,
        settings_path,
        vk_path,
        
    )

assert res == True
print("verified")

verified
